## Notebook 05c -- Rung 4 v2: weighted refit
Rung 4 v2 = retrain the rung-3 model on the 9 training participants **plus session 1 of the held-out participant**. Session 1 gets the weight `w = N_pool / N_enrol`, so it counts as much as all 9 other people together. A control arm repeats the refit with `w = 1`, to separate "personal data" from "more rows". Plan: `docs/logs/20261008_140743_rung4_v2_plan.md`.
Both arms use each participant's rung-3 `best_params` unchanged. Test sessions are 2 to K. This notebook only produces numbers (`fold_metrics_rung4_v2.json`). Statistics are Phase 3, in `significance_analysis.py`.

Load the shared data (same `FEATURES`, `FEATURE_COLS`, `SUBJECT_FOLDS` as 03/04/05).

In [ ]:
from _common import *

Imports and device. Same stack as `inference_timing.py`.

In [ ]:
import gc
import time
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

Switches. `SMOKE = True` runs Ridge only, on participants 1 and 10 (the two with a missing session), and writes to a separate file so the real output is never touched.

In [ ]:
SMOKE = False

OUT_PATH = CURRENT_FOLDER / ('fold_metrics_rung4_v2_smoke.json' if SMOKE else 'fold_metrics_rung4_v2.json')
SMOKE_PARTICIPANTS = [1, 10]
ENROL_SESSION = 1
print('SMOKE:', SMOKE, '->', OUT_PATH.name)

Model builders, copied from `inference_timing.py` unchanged (which copied 03 / 04 / 05). Ridge and XGBoost first.

In [ ]:
def make_ridge(params):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=params['alpha'])),
    ])


def make_xgboost(params):
    return XGBRegressor(
        **params,
        random_state=RANDOM_SEED,
        device='cuda',
        tree_method='hist',
        n_jobs=-1,
    )

The MLP builder: a small PyTorch network wrapped in a fit / predict class, trained on GPU. **One change from 05b:** `fit` takes an optional `sample_weight`. With weights, the batch loss is `sum(w * err^2) / sum(w)`. Without weights, it runs the unchanged rung-3 code path (`nn.MSELoss()`), so the rung-3 refit check still reproduces exactly.

In [ ]:
class MLPTorch(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRegressor:
    def __init__(self, hidden_dims, dropout, lr, weight_decay, batch_size, n_epochs, random_state=RANDOM_SEED):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.model_ = MLPTorch(X.shape[1], self.hidden_dims, self.dropout).to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()

        X_t = torch.from_numpy(X).to(DEVICE)
        y_t = torch.from_numpy(y).to(DEVICE)
        w_t = None if sample_weight is None else torch.from_numpy(np.asarray(sample_weight, dtype=np.float32)).to(DEVICE)
        n = len(X_t)

        self.model_.train()
        for epoch in range(self.n_epochs):
            perm = torch.randperm(n, device=DEVICE)
            for start in range(0, n, self.batch_size):
                idx = perm[start:start + self.batch_size]
                opt.zero_grad()
                pred = self.model_(X_t[idx])
                if w_t is None:
                    loss = loss_fn(pred, y_t[idx])
                else:
                    w_b = w_t[idx]
                    loss = (w_b * (pred - y_t[idx]) ** 2).sum() / w_b.sum()   # weighted MSE, equals MSELoss when all w = 1
                loss.backward()
                opt.step()
        del X_t, y_t, w_t
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        device = next(self.model_.parameters()).device
        X_t = torch.from_numpy(X).to(device)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(X_t)
        return pred.cpu().numpy()


def hidden_dims_from(width, n_layers):
    dims = []
    w = width
    for _ in range(n_layers):
        dims.append(max(w, 8))
        w = max(w // 2, 8)
    return dims


def make_mlp(params):
    hidden_dims = hidden_dims_from(params['width'], params['n_layers'])
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_dims=hidden_dims,
            dropout=params['dropout'],
            lr=params['lr'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            n_epochs=params['n_epochs'],
            random_state=RANDOM_SEED,
        )),
    ])

Identity check for the weighted loss: with all `w = 1`, the weighted MSE must equal `nn.MSELoss()`.

In [ ]:
_pred = torch.randn(1024)
_true = torch.randn(1024)
_w = torch.ones(1024)
_weighted = float((_w * (_pred - _true) ** 2).sum() / _w.sum())
_plain = float(nn.MSELoss()(_pred, _true))
assert abs(_weighted - _plain) < 1e-6, (_weighted, _plain)
print('weighted MSE with w = 1:', _weighted, '| nn.MSELoss:', _plain)

How the weight reaches each model. Ridge and the MLP are `Pipeline`s, so the scaler and the model both get the weight through the step prefix (the scaler uses the same weights as the model). XGBoost is a bare `XGBRegressor` with no scaler, so it takes `sample_weight` directly.

In [ ]:
def fit_weighted(model_name, model, X, y, w):
    if model_name == 'Ridge':
        model.fit(X, y, scaler__sample_weight=w, ridge__sample_weight=w)
    elif model_name == 'XGBoost':
        model.fit(X, y, sample_weight=w)
    else:
        model.fit(X, y, scale__sample_weight=w, mlp__sample_weight=w)
    return model

The three models and their rung-3 result files. In smoke mode only Ridge stays.

In [ ]:
MODELS = {
    'Ridge': ('fold_metrics_ridge_rung3.json', make_ridge),
    'XGBoost': ('fold_metrics_xgboost_rung3.json', make_xgboost),
    'MLP': ('fold_metrics_mlp_rung3.json', make_mlp),
}
if SMOKE:
    MODELS = {'Ridge': MODELS['Ridge']}

print(list(MODELS))

Session 1 is the enrolment session, so it must exist for every participant. Participant 001 has no session 4 and participant 010 has no session 6, so the session lists come from the manifest, never from `range(1, 11)`.

In [ ]:
SESSIONS = {int(p.removeprefix('participant')): sorted(v['sessions']) for p, v in MANIFEST['participants'].items()}
for p, sess in SESSIONS.items():
    assert 1 in sess, p
print(SESSIONS)

Resume support. If the output file exists, load it and skip every (model, participant) pair already in it. The file is rewritten after each pair (both arms together).

In [ ]:
if OUT_PATH.exists():
    with open(OUT_PATH) as f:
        records = json.load(f)
else:
    records = []

done = {(r['model'], r['participant_id']) for r in records}
print(len(records), 'records already on disk;', len(done), 'pairs done')

Main loop. For each model and held-out participant:
1. Refit the rung-3 model on the 9 training participants with the stored `best_params`. Check that it reproduces the stored RMSE to 4 decimals. Stop if it does not.
2. Build the v2 training set: pool rows + session-1 rows of the held-out participant. Check the leakage rules.
3. Ridge only: refit with `w = 0` on the session-1 rows. It must equal the rung-3 refit to 4 decimals.
4. Train the **refit** arm (`w = N_pool / N_enrol`) and the **control** arm (`w = 1`), same `best_params`, same seed.
5. Score both arms on sessions 2 to K, one entry per session.

In [ ]:
def score_sessions(pred, y, sess, test_sessions):
    entries = []
    for s in test_sessions:
        m = sess == s
        err = pred[m] - y[m]
        entries.append({
            'session': int(s),
            'n': int(m.sum()),
            'rmse': float(np.sqrt(np.mean(err ** 2))),
            'mae': float(np.mean(np.abs(err))),
        })
    return entries


def pooled_rmse(entries):
    return float(np.sqrt(sum(e['n'] * e['rmse'] ** 2 for e in entries) / sum(e['n'] for e in entries)))


for model_name, (json_name, make_model) in MODELS.items():
    with open(CURRENT_FOLDER / json_name) as f:
        fold_records = {r['participant_id']: r for r in json.load(f)}

    for fold in SUBJECT_FOLDS:
        test_p = int(fold['test_participant'].removeprefix('participant'))
        if SMOKE and test_p not in SMOKE_PARTICIPANTS:
            continue
        if (model_name, test_p) in done:
            continue
        record = fold_records[test_p]
        params = record['best_params']
        json_rmse = record['metrics']['overall']['rmse']
        test_sessions = [s for s in SESSIONS[test_p] if s != ENROL_SESSION]

        train_nums = [int(p.removeprefix('participant')) for p in fold['train_participants']]
        pool_mask = FEATURES['participant_id'].isin(train_nums)
        held_mask = FEATURES['participant_id'] == test_p
        enrol_mask = held_mask & (FEATURES['session_id'] == ENROL_SESSION)
        test_mask = held_mask & (FEATURES['session_id'] != ENROL_SESSION)
        assert test_p not in train_nums                                            # leakage: held-out participant not in the pool
        assert not (FEATURES.loc[pool_mask, 'participant_id'] == test_p).any()     # leakage: no held-out rows in the pool

        # 1. rung-3 refit check (pool only, unweighted, the rung-3 code path)
        X_all = FEATURES.loc[held_mask, FEATURE_COLS].to_numpy()
        y_all = FEATURES.loc[held_mask, 'pct_mvc_corr'].to_numpy()
        X_pool = FEATURES.loc[pool_mask, FEATURE_COLS].to_numpy()
        y_pool = FEATURES.loc[pool_mask, 'pct_mvc_corr'].to_numpy()
        t0 = time.time()
        model3 = make_model(params)
        model3.fit(X_pool, y_pool)
        fit3_s = time.time() - t0
        del X_pool, y_pool
        gc.collect()
        refit_rmse = float(np.sqrt(np.mean((np.asarray(model3.predict(X_all), dtype=float) - y_all) ** 2)))
        if round(refit_rmse, 4) != round(json_rmse, 4):
            raise RuntimeError(f'{model_name} p{test_p}: refit rmse {refit_rmse:.6f} != json rmse {json_rmse:.6f}')
        del X_all, y_all

        # 2. v2 training set: pool + session-1 rows, in FEATURES order
        train_mask = pool_mask | enrol_mask
        assert not (train_mask & test_mask).any()                                  # leakage: no test session in training
        assert set(FEATURES.loc[train_mask & held_mask, 'session_id'].unique()) == {ENROL_SESSION}   # enrolment = session 1 only
        is_enrol = enrol_mask[train_mask].to_numpy()
        X_train = FEATURES.loc[train_mask, FEATURE_COLS].to_numpy()
        y_train = FEATURES.loc[train_mask, 'pct_mvc_corr'].to_numpy()
        X_test = FEATURES.loc[test_mask, FEATURE_COLS].to_numpy()
        y_test = FEATURES.loc[test_mask, 'pct_mvc_corr'].to_numpy()
        sess_test = FEATURES.loc[test_mask, 'session_id'].to_numpy()
        assert sorted(np.unique(sess_test).tolist()) == test_sessions
        n_enrol = int(is_enrol.sum())
        n_pool = int((~is_enrol).sum())
        w_refit = n_pool / n_enrol

        pred3_test = np.asarray(model3.predict(X_test), dtype=float)
        rmse3 = float(np.sqrt(np.mean((pred3_test - y_test) ** 2)))
        del model3
        gc.collect()

        # 3. Ridge only: w = 0 on the session-1 rows must reproduce the rung-3 refit
        if model_name == 'Ridge':
            w0 = np.where(is_enrol, 0.0, 1.0)
            model0 = fit_weighted(model_name, make_model(params), X_train, y_train, w0)
            rmse0 = float(np.sqrt(np.mean((np.asarray(model0.predict(X_test), dtype=float) - y_test) ** 2)))
            if round(rmse0, 4) != round(rmse3, 4):
                raise RuntimeError(f'Ridge p{test_p}: w=0 rmse {rmse0:.6f} != rung-3 refit rmse {rmse3:.6f}')
            del model0

        # 4. + 5. refit arm and control arm
        new_records = []
        for arm, w_enrol in [('refit', w_refit), ('control', 1.0)]:
            w = np.where(is_enrol, w_enrol, 1.0)
            t0 = time.time()
            model = fit_weighted(model_name, make_model(params), X_train, y_train, w)
            fit_s = time.time() - t0
            pred = np.asarray(model.predict(X_test), dtype=float)
            new_records.append({
                'model': model_name,
                'participant_id': int(test_p),
                'arm': arm,
                'enrol_session': ENROL_SESSION,
                'w': float(w_enrol),
                'n_pool': n_pool,
                'n_enrol': n_enrol,
                'rung3_refit_rmse': refit_rmse,
                'json_rmse': float(json_rmse),
                'fit_s': round(fit_s, 1),
                'test': score_sessions(pred, y_test, sess_test, test_sessions),
            })
            del model, pred
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        records.extend(new_records)
        with open(OUT_PATH, 'w') as f:
            json.dump(records, f, indent=2)
        done.add((model_name, test_p))

        print(f"{model_name:8s} p{test_p:02d}  rung3 check ok ({fit3_s:.0f}s)  w={w_refit:5.1f}  s2..sK: "
              f"rung3={rmse3:6.2f}  control={pooled_rmse(new_records[1]['test']):6.2f}  refit={pooled_rmse(new_records[0]['test']):6.2f}")

        del X_train, y_train, X_test, y_test, pred3_test
        gc.collect()

print(f"done -- {len(records)} records in {OUT_PATH.name}")

Per-participant table, one block per model (participants are rows). Pooled RMSE over sessions 2 to K for rung 1, rung 3, the control and the refit. Rung 3 comes from `fold_metrics_rung4.json` (turn `k = 1`), rung 1 from the rung-1 files. No tests here: statistics are Phase 3. It reads the JSON, so it also works after a restart.

In [ ]:
def pooled(n, rmse):
    n = np.asarray(n, dtype=float)
    rmse = np.asarray(rmse, dtype=float)
    return float(np.sqrt((n * rmse ** 2).sum() / n.sum()))


with open(OUT_PATH) as f:
    v2 = json.load(f)
with open(CURRENT_FOLDER / 'fold_metrics_rung4.json') as f:
    v1 = [r for r in json.load(f) if r['enrol_session'] == ENROL_SESSION]
RUNG1_FILES = {'Ridge': 'fold_metrics_ridge.json', 'XGBoost': 'fold_metrics_xgboost.json', 'MLP': 'fold_metrics_mlp.json'}

for model_name in MODELS:
    if not any(r['model'] == model_name for r in v2):
        continue
    with open(CURRENT_FOLDER / RUNG1_FILES[model_name]) as f:
        r1 = {(r['participant_id'], r['test_session']): r['metrics']['overall'] for r in json.load(f)}
    rows = []
    for p in sorted({r['participant_id'] for r in v2 if r['model'] == model_name}):
        arms = {r['arm']: r for r in v2 if r['model'] == model_name and r['participant_id'] == p}
        tests = arms['refit']['test']
        sess = [t['session'] for t in tests]
        v1_tests = {t['session']: t for t in next(r for r in v1 if r['model'] == model_name and r['participant_id'] == p)['test']}
        assert [t['session'] for t in arms['control']['test']] == sess
        assert all(v1_tests[s]['n'] == t['n'] == r1[(p, s)]['n'] for s, t in zip(sess, tests))   # same frames in every rung
        rows.append({
            'participant': f'p{p:03d}',
            'w': arms['refit']['w'],
            'rung1': pooled([r1[(p, s)]['n'] for s in sess], [r1[(p, s)]['rmse'] for s in sess]),
            'rung3': pooled([v1_tests[s]['n'] for s in sess], [v1_tests[s]['rmse_rung3'] for s in sess]),
            'control': pooled([t['n'] for t in arms['control']['test']], [t['rmse'] for t in arms['control']['test']]),
            'refit': pooled([t['n'] for t in tests], [t['rmse'] for t in tests]),
        })
    table = pd.DataFrame(rows)
    print(model_name)
    print(table.round(2).to_string(index=False))
    print('mean', table[['rung1', 'rung3', 'control', 'refit']].mean().round(2).to_dict(),
          '| refit < rung3 in', int((table['refit'] < table['rung3']).sum()), 'of', len(table))
    print()